# Phase 3 — Worker, one molecule at a time

**Goal.** Open up the per-molecule pipeline and watch each step: a 1D SMILES string becomes a 3D structure, gets relaxed with a neural-network potential, and finally a DFT calculation tells you where the frontier orbitals sit.

This is the scientifically interesting notebook. The other three are plumbing — this one is chemistry.

## How to run it — *must* be inside the container on a GPU node

Unlike the other notebooks, this one calls into CUDA kernels (AIMNet2 inference + gpu4pyscf DFT) that don't exist on the head node. You need two things:

1. A GPU allocation.
2. A Jupyter kernel running *inside* the shared container.

```bash
# 1. grab an interactive GPU session
salloc --partition=kempner_dev --account=kempner_dev \
       --gres=gpu:1 --cpus-per-task=4 --mem=32G --time=1:00:00

# 2. start Jupyter inside the container (run from alchemi-demo/)
singularity exec --nv \
    --env SSL_CERT_FILE=/etc/ssl/certs/ca-certificates.crt \
    --env CURL_CA_BUNDLE=/etc/ssl/certs/ca-certificates.crt \
    --env REQUESTS_CA_BUNDLE=/etc/ssl/certs/ca-certificates.crt \
    --bind $PWD/data:/data --bind $PWD/tutorial:/tutorial \
    --bind /n/netscratch/kempner_dev/Lab/bdesinghu/Agent/alchemi/container/aimnet_assets:/aimnet_assets:ro \
    /n/holylfs06/LABS/kempner_shared/Everyone/containers/applications/alchemi-ht/alchemi_ht.sif \
    jupyter lab --ip=0.0.0.0 --port=8888 --no-browser
```

Then SSH port-forward `8888` from the compute node to your laptop (`ssh -L 8888:<node>:8888 holylogin`) and paste the token URL into your browser.

**Sanity check:** if you can't `import torch; torch.cuda.is_available()` → `True` in the first cell, stop and fix the environment. Nothing below will work.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA — are you inside the container on a GPU node?'
print('CUDA device:', torch.cuda.get_device_name(0))

## Pick a molecule

We'll use a small GDB-17 example — a brominated polycyclic heterocycle. Swap in any SMILES string you like and re-run; the pipeline doesn't care what you throw at it as long as RDKit can parse it and AIMNet2 was trained on its elements (H, C, N, O, F, S, Cl, Br, P, I).

In [ ]:
smiles = 'BrC1=C2C3=C4C(CC3CCC2=O)C(=N)NC4=N1'
smiles

## Step A — 3D embedding with RDKit ETKDGv3

**What happens.** RDKit reads the SMILES (connectivity + bond orders), adds explicit hydrogens (SMILES typically omits them), then uses **ETKDG** — a distance-geometry / knowledge-based hybrid — to generate one 3D conformer. ETKDGv3 is the current recommended variant; it uses improved torsion preferences learned from crystallographic data.

**Why follow it with MMFF?** Pure ETKDG produces a *reasonable* geometry but not a *minimized* one. A quick MMFF94 force-field cleanup (cheap, classical, 200 iterations max) gets the atoms near a local minimum so that AIMNet2 in Step B has a sensible starting point. Skip MMFF and you sometimes see the neural relaxation thrash for many steps before settling.

**Random seed.** We fix `randomSeed=0xC0FFEE` so the same SMILES always produces the same geometry. Drop that line to get conformer diversity.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)
params = AllChem.ETKDGv3()
params.randomSeed = 0xC0FFEE
assert AllChem.EmbedMolecule(mol, params) == 0, 'embedding failed'
AllChem.MMFFOptimizeMolecule(mol, maxIters=200)

print(f'n_atoms (with Hs) = {mol.GetNumAtoms()}')
print('\nFirst 5 atomic coords (Å):')
conf = mol.GetConformer()
for i in range(min(5, mol.GetNumAtoms())):
    a = mol.GetAtomWithIdx(i)
    p = conf.GetAtomPosition(i)
    print(f'  {a.GetSymbol():2s}  {p.x:8.3f} {p.y:8.3f} {p.z:8.3f}')

## Step B — AIMNet2 relaxation with ASE FIRE

**What AIMNet2 is.** A neural-network interatomic potential trained to reproduce DFT energies and forces for small organic molecules. It gives you ~DFT accuracy at ~classical-force-field speed — perfect for geometry relaxation where you'd otherwise burn CPU-hours on BFGS-with-DFT steps.

**Why this step at all?** We *could* hand the MMFF-relaxed geometry straight to gpu4pyscf. But DFT energies are exquisitely sensitive to geometry near the equilibrium — a few hundredths of an Ångström off and your HOMO/LUMO can shift by tenths of an eV. Relaxing first with a cheap-but-accurate potential pays for itself.

**Why ASE FIRE and not BFGS?** FIRE (Fast Inertial Relaxation Engine) is robust to noisy forces (NN potentials have a little noise) and rarely blows up. BFGS converges faster when it works but can overshoot on badly-conditioned systems. For a screen you want robustness > speed.

**Why `needs_dispersion=False`?** Container-level bug: the aimnet package's external DFT-D3 integration has a signature mismatch against the shipped `nvalchemiops` (`TypeError: dftd3() got an unexpected keyword argument 'cell'`). Disabling external D3 loses <1 kcal/mol of accuracy — fine for triage screens, revisit if you need publication-grade absolute energies.

In [ ]:
from ase import Atoms
from ase.optimize import FIRE
from aimnet.calculators import AIMNet2ASE
from aimnet.calculators.calculator import AIMNet2Calculator

symbols = [a.GetSymbol() for a in mol.GetAtoms()]
positions = mol.GetConformer().GetPositions()
atoms = Atoms(symbols=symbols, positions=positions)

base = AIMNet2Calculator('/aimnet_assets/aimnet2_wb97m_d3_0.pt', needs_dispersion=False)
atoms.calc = AIMNet2ASE(base)

FIRE(atoms, logfile=None).run(fmax=0.05, steps=200)
print(f'relaxed potential energy (eV): {atoms.get_potential_energy():.4f}')
print(f'max |force| (eV/Å): {abs(atoms.get_forces()).max():.4f}  (target: < 0.05)')

The energy printed here is the AIMNet2 potential-energy surface value — useful as a proxy but **not** what we'll report. The DFT energy from Step C is the ground truth. `fmax` should be below the 0.05 eV/Å threshold we asked for; if it isn't, the optimizer ran out of steps (bump `steps=200` up).

## Step C — HOMO/LUMO via gpu4pyscf

**What it is.** A single-point DFT calculation at the relaxed geometry. We use the **B3LYP** hybrid functional — still the most widely cited workhorse for organic molecules — with **def2-SVP**, a split-valence polarization basis. This combination trades some accuracy for a big speedup versus def2-TZVP; for HOMO/LUMO *gap* predictions it's adequate.

**What HOMO/LUMO mean.**

- **HOMO** (Highest Occupied Molecular Orbital) — the highest-energy orbital that has electrons in it. Its energy is roughly the negative of the ionization potential (Koopmans' theorem).
- **LUMO** (Lowest Unoccupied Molecular Orbital) — the lowest-energy empty orbital. Its energy is roughly the negative of the electron affinity.
- **Gap = LUMO − HOMO** — a chemical stability and reactivity descriptor. Large gap → kinetically inert (hard to excite electrons into a reactive state). Small gap → easy to oxidize/reduce, often colored, often reactive.

**Units.** PySCF reports MO energies in **Hartree**. We convert to **eV** (1 Ha ≈ 27.2114 eV) because chemists think in eV. Total energies stay in Hartree because they span large numbers.

In [ ]:
from pyscf import gto
from gpu4pyscf import dft

atom_spec = [(a.symbol, tuple(float(x) for x in a.position)) for a in atoms]
gpu_mol = gto.M(atom=atom_spec, basis='def2-svp', charge=0, spin=0, verbose=0)
mf = dft.RKS(gpu_mol, xc='B3LYP')
energy_hartree = float(mf.kernel())

HA_TO_EV = 27.211386245988
occupied = mf.mo_energy[mf.mo_occ > 0]
virtual  = mf.mo_energy[mf.mo_occ == 0]
homo_ev = float(occupied[-1]) * HA_TO_EV
lumo_ev = float(virtual[0])  * HA_TO_EV

print(f'energy = {energy_hartree:.4f} Hartree   ({energy_hartree * HA_TO_EV:.1f} eV)')
print(f'HOMO   = {homo_ev:.3f} eV')
print(f'LUMO   = {lumo_ev:.3f} eV')
print(f'gap    = {lumo_ev - homo_ev:.3f} eV')

### Interpreting the numbers

For this aromatic bromoheterocycle, a gap in the ~4–6 eV range is typical. For reference:

| Compound class | Typical B3LYP/def2-SVP gap |
| --- | --- |
| Alkanes (saturated, inert) | 8–10 eV |
| Benzene | ~6.7 eV |
| Medium aromatics (like this one) | 4–6 eV |
| Pigments / dyes | 2–3 eV |
| Small-gap semiconducting polymers | 1–2 eV |

A gap in the mid-single-digit eV range says the molecule is likely a stable, potentially chromophoric organic compound — reasonable screening material.

## Tying it together

You just did — cell by cell — what the production loop in `3_worker_node.py` does for every row in every chunk. The production version wraps each molecule in `try/except` so a single bad SMILES or convergence failure doesn't sink an entire chunk; the failure is logged and the loop moves on.

**Try it yourself.** Go back to the SMILES cell and paste another molecule — a simple alkane, an aromatic dye, a drug. Watch the gap change accordingly.

**Next:** open `04_aggregate_and_analyze.ipynb` to see what the full dataset looks like in aggregate.